In [ ]:
# Install Cadabra if necessary, and enable Cadabra input in this notebook.
try:
    import cadabra2
except ImportError:
    %pip install -q cadabra2
%load_ext cadabra2_jupyter

# Friedmann-Robertson-Walker

This example starts from the Friedmann-Robertson-Walker metric for a homogeneous and isotropic universe, to derive the Friedmann equations. We start by declaring indices. We also introduce some shorthand notations to display derivatives using dots.

In [ ]:
__cdbkernel__.display_fractions=True

In [ ]:
{t, r, \theta, \phi}::Coordinate();
{\mu,\nu,\rho,\sigma,\kappa,\lambda}::Indices(values={t, r, \theta, \phi}, position=fixed);

In [ ]:
a::Depends(t);
\partial{#}::PartialDerivative;
\partial_{t}{A??}::LaTeXForm("\dot{", A??, "}");
\partial_{t t}{A??}::LaTeXForm("\ddot{", A??, "}");

Now it is time to declare the metric and its components.

In [ ]:
g_{\mu\nu}::Metric;
g^{\mu\nu}::InverseMetric;

In [ ]:
metric:= { g_{t t} = -1, 
           g_{r r} = a**2 / (1 - k r**2), 
           g_{\theta\theta} = a**2 r**2, 
           g_{\phi\phi} = a**2 r**2 \sin(\theta)**2 };

The inverse metric is computed by using `complete`,

In [ ]:
complete(metric, $g^{\mu\nu}$);

Now we need the Christoffel symbols, and then the Riemann tensor. First define them,

In [ ]:
chr:= \Gamma^{\sigma}_{\mu\nu} = 1/2 g^{\sigma\rho} ( \partial_{\mu}{g_{\nu\rho}} + \partial_{\nu}{g_{\mu\rho}}
                                                           - \partial_{\rho}{g_{\mu\nu}} );

and then evaluate the components for the particular component values of the FRW metric:

In [ ]:
evaluate(chr, metric, rhsonly=True);

In [ ]:
riem:= R^{\rho}_{\sigma\mu\nu} = \partial_{\mu}{\Gamma^{\rho}_{\sigma\nu}} - \partial_{\nu}{\Gamma^{\rho}_{\sigma\mu}}
                                 + \Gamma^{\rho}_{\mu\lambda} \Gamma^{\lambda}_{\nu\sigma}
                                 - \Gamma^{\rho}_{\nu\lambda} \Gamma^{\lambda}_{\mu\sigma};

In [ ]:
substitute(riem, chr)

In [ ]:
evaluate(riem, metric, rhsonly=True);

Finally, the Ricci tensor and after that the Ricci scalar, which we need to compute the Einstein tensor shortly.

In [ ]:
ricc:= R_{\mu\nu} = R^{\rho}_{\mu\nu\rho};

In [ ]:
substitute(ricc, riem)
evaluate(ricc, metric, rhsonly=True);

In [ ]:
rsc:= R = g^{\mu\nu} R_{\mu\nu};

In [ ]:
substitute(rsc, ricc)
evaluate(rsc, metric, rhsonly=True);

Let's compute the Einstein tensor, as we will need that to compute equations of motion for the scale factor. Start with the definition,

In [ ]:
einstein:= G_{\mu\nu} = R_{\mu\nu} - 1/2 g_{\mu\nu} R;

Substitute the components for the Ricci tensor and scalar computed above, and then use the metric at the last step to finish the computation,

In [ ]:
substitute(einstein, ricc)
substitute(einstein, rsc)
evaluate(einstein, metric, rhsonly=True);

Let's compute the Einstein tensor with mixed indices by simply raising one:

In [ ]:
evaluate(substitute($G^{\mu}_{\nu} = g^{\mu\rho} G_{\rho\nu}$, einstein), metric, rhsonly=True);

In [ ]:
einsteinmixed:= G^{\mu}_{\nu} = g^{\mu\rho} G_{\rho\nu};

In [ ]:
substitute(einsteinmixed, einstein)
evaluate(einsteinmixed, metric, rhsonly=True);

Now on to the Friedmann equations. We need the Einstein equations first (could have used the pre-computed Einstein tensor above),

In [ ]:
frw:= R_{\mu\nu} - 1/2 g_{\mu\nu} = T_{\mu\nu};

The stress tensor depends on the density $\rho$ and the pressure $p$,

In [ ]:
stress:= { T_{t t} = \rho, T_{r r} = p g_{r r}, T_{\phi\phi} = p g_{\phi\phi}, T_{\theta\theta} = p g_{\theta\theta} };

In [ ]:
substitute(frw, ricc);

In [ ]:
evaluate(frw, join(metric, stress));

In [ ]:
substitute(frw, metric);

This is still a bit ugly; we would like to move the right-hand sides to the left-hand side so we have four expressions that need to vanish for the Einstein equations to hold. Use the `cdb.core.manip` module for that, which contains `lhs` and `rhs` functions:

In [ ]:
from cdb.core.manip import *

In [ ]:
q=lhs(frw) - rhs(frw);

Now we can collect the components into one tensor,

In [ ]:
collect_components(q);

If you want to extract the individual components, use the `get_component` function of the `cdb.core.component`
package:

In [ ]:
from cdb.core.component import *

In [ ]:
e1=get_component(q, $\theta, \theta$);

In [ ]:
e2=get_component(q, $t, t$);